# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

In [1]:
import open3d as o3d
import numpy as np
import copy

In [2]:
def draw_registration(source, target, transformation=None, recolor=False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if recolor:
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if transformation is not None:
        source_temp.transform(transformation)

    o3d.visualization.draw_geometries([source_temp, target_temp])

def get_cloud_points_and_calculate_features(path1, path2):
    source = o3d.io.read_point_cloud(path1)
    target = o3d.io.read_point_cloud(path2)

    voxel_size = 0.05
    source = source.voxel_down_sample(voxel_size=voxel_size)
    target = target.voxel_down_sample(voxel_size=voxel_size)

    source.estimate_normals()
    target.estimate_normals()

    feature_source = o3d.pipelines.registration.compute_fpfh_feature(
        source, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
    )
    feature_target = o3d.pipelines.registration.compute_fpfh_feature(
        target, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*5, max_nn=100)
    )

    return source, target, feature_source, feature_target

In [3]:
source, target, feature_source, feature_target = get_cloud_points_and_calculate_features("global_registration/r1.pcd", "global_registration/r2.pcd")

distance_threshold = 0.05
point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()
point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint()
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source, target, feature_source, feature_target,
    True, distance_threshold, point_to_point
)

draw_registration(source, target, ransac_result.transformation, True)

In [4]:
#Task 2a
source, target, feature_source, feature_target = get_cloud_points_and_calculate_features('global_registration/car1.ply', 'global_registration/car2.ply')
draw_registration(source, target, recolor=True)

In [5]:
#global matching moving points closer to each other
distance_threshold = 0.05
point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint()

ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source, target, feature_source, feature_target, True,
    distance_threshold, point_to_point
)

draw_registration(source, target, ransac_result.transformation, recolor=True)

In [6]:
#local matching 
threshold = 0.02
icp_result = o3d.pipelines.registration.registration_icp(
    source, target, threshold, ransac_result.transformation,
    point_to_point
)

draw_registration(source, target, icp_result.transformation, recolor=True)

Task 2b
Function for calculating cloud points of the car. It Takes 2 arguments first one is step(default value is 1) which will skip this amount of frames. We skip images to reduce computation time in trade for accuracy. The second argument is number of images (default value is 1722) in dataset. It is used to indicate number of image we want to traverse and include in calculations. What is worth mentioning is that voxel_size is used in different functions and it is tuned accordingly to trial and error steps.

In [ ]:

def retrieve_point_cloud(step=1, number_of_images=1722):
    voxel_size = 0.05
    camera = o3d.camera.PinholeCameraIntrinsic(
        o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault
    )

    #function to read point cloud from rgb image and depth
    def load_point_cloud(idx):
        img = o3d.io.read_image(f'car_challange/rgb/{idx:07d}.jpg')
        depth = o3d.io.read_image(f'car_challange/depth/{idx:07d}.png')
        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
            img, depth, convert_rgb_to_intensity=False
        )
        pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, camera)
        #rotating point cloud so it is not rotated
        pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
        pcd = pcd.voxel_down_sample(voxel_size)
        pcd.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
        )
        return pcd

    point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()
    point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint()

    #loading first point cloud from the dataset so it can be later compared with next point cloud
    prev_cloud = load_point_cloud(1)
    global_pose = np.identity(4)
    #variable which will store final result, deepcopy is used so every operation on accumulated doesn't change prev_cloud
    accumulated = copy.deepcopy(prev_cloud)
    icp_fitness_threshold = 0.8   #if the first local matching has bad result this threshold starts global alignment first to improve result


    for idx in range(2, number_of_images + 1, step):
        print(f"Frame {idx}")
        #loading currect point cloud so it can be comared with previous point cloud
        current_cloud = load_point_cloud(idx)

        icp_result = o3d.pipelines.registration.registration_icp(
            current_cloud, prev_cloud, voxel_size * 0.8, np.identity(4), point_to_plane
        )

        if icp_result.fitness < icp_fitness_threshold:
            print(f"frame {idx} requires global matching (fitness={icp_result.fitness:.3f})")

            feature_current = o3d.pipelines.registration.compute_fpfh_feature(
                current_cloud, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 10, max_nn=100)
            )
            feature_prev = o3d.pipelines.registration.compute_fpfh_feature(
                prev_cloud, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 10, max_nn=100)
            )

            ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
                current_cloud, prev_cloud, feature_current, feature_prev,
                True, voxel_size * 2, point_to_point, 3, [],
                o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999)
            )
            icp_result = o3d.pipelines.registration.registration_icp(
                current_cloud, prev_cloud, voxel_size * 0.8, ransac_result.transformation, point_to_plane
            )

        # Composing transformation in order not to do transformation from the beginning but the latest point
        global_pose = global_pose @ icp_result.transformation

        current_world = copy.deepcopy(current_cloud)
        current_world.transform(global_pose)

        #accumulating new point cloud in our final result variable
        accumulated += current_world
        #down sampling new result for better performence
        accumulated = accumulated.voxel_down_sample(voxel_size)

        #Chaning prev to current for further calculations
        prev_cloud = current_cloud

    return accumulated

In [ ]:
accumulated_cloud = retrieve_point_cloud(step=5, number_of_images=1722)

#Visualizing final result and saving it to the file
o3d.visualization.draw_geometries([accumulated_cloud])
o3d.io.write_point_cloud("car_reconstruction.ply", accumulated_cloud)

Frame 2
Frame 7
Frame 12
Frame 17
Frame 22
Frame 27
Frame 32
Frame 37
Frame 42
Frame 47
Frame 52
Frame 57
Frame 62
Frame 67
Frame 72
Frame 77
Frame 82
Frame 87
Frame 92
Frame 97
Frame 102
Frame 107
Frame 112
Frame 117
Frame 122
Frame 127
Frame 132
Frame 137
Frame 142
Frame 147
Frame 152
Frame 157
Frame 162
Frame 167
Frame 172
Frame 177
Frame 182
Frame 187
Frame 192
Frame 197
Frame 202
Frame 207
Frame 212
Frame 217
Frame 222
Frame 227
Frame 232
Frame 237
Frame 242
Frame 247
Frame 252
Frame 257
Frame 262
Frame 267
Frame 272
Frame 277
Frame 282
Frame 287
Frame 292
Frame 297
Frame 302
Frame 307
Frame 312
Frame 317
Frame 322
Frame 327
Frame 332
Frame 337
Frame 342
Frame 347
frame 347 requires global matching (fitness=0.784)
Frame 352
Frame 357
Frame 362
Frame 367
Frame 372
Frame 377
Frame 382
Frame 387
Frame 392
Frame 397
Frame 402
Frame 407
Frame 412
Frame 417
Frame 422
Frame 427
Frame 432
Frame 437
Frame 442
Frame 447
Frame 452
Frame 457
Frame 462
Frame 467
Frame 472
Frame 477
Frame 482
F

True